In [3]:
from openpyxl import load_workbook
import pandas as pd

# ==========================
# Load Workbook
# ==========================
wb = load_workbook("Crop_Yield_Dataset.xlsx", data_only=True)
ws = wb.active

# --------------------------
# Read 3 header rows
# --------------------------
crop_row = [ws.cell(1, c).value for c in range(1, ws.max_column + 1)]
# Read season row
season_values = [ws.cell(2, c).value for c in range(1, ws.max_column + 1)]

# Forward-fill merged season cells
season_row = []
current_season = None

for value in season_values:
    if value is not None:
        current_season = value
    season_row.append(current_season)
metric_values = [ws.cell(3, c).value for c in range(1, ws.max_column + 1)]

metric_row = []
current_metric = None

for value in metric_values:
    if value is not None:
        current_metric = value
    metric_row.append(current_metric)

# Forward-fill crop names
crop_names = []
current_crop = None

for value in crop_row:
    if value is not None:
        current_crop = value
    crop_names.append(current_crop)

records = []

# ==========================
# Read Data
# ==========================
for r in range(4, ws.max_row + 1):

    state = ws.cell(r, 1).value
    district = ws.cell(r, 2).value
    year = ws.cell(r, 3).value

    if state is None:
        state = records[-1]["State"]

    if district is None:
        district = records[-1]["District"]

    crop_dict = {}

    for c in range(4, ws.max_column + 1):

        crop = crop_names[c - 1]
        season = season_row[c - 1]
        metric = metric_row[c - 1]

        key = (crop, season)

        if key not in crop_dict:
            crop_dict[key] = {
                "State": str(state).replace("1. ", ""),
                "District": str(district).replace("1. ", ""),
                "Year": year,
                "Crop": crop,
                "Season": season,
                "Area": None,
                "Production": None,
                "Yield": None,
            }

        value = ws.cell(r, c).value

        metric = str(metric).lower()

        if "area" in metric:
            crop_dict[key]["Area"] = value

        elif "production" in metric:
            crop_dict[key]["Production"] = value

        elif "yield" in metric:
            crop_dict[key]["Yield"] = value

    records.extend(crop_dict.values())

# ==========================
# Create DataFrame
# ==========================
df = pd.DataFrame(records)

# Remove completely empty rows
df = df.dropna(subset=["Area", "Production", "Yield"], how="all")

# Save
df.to_csv("Crop_Yield_Long_Format.csv", index=False)

print(df.head())

print("\nRows :", len(df))
print("Districts :", df["District"].nunique())
print("Crops :", df["Crop"].nunique())
print("Seasons :", df["Season"].unique())

        State  District         Year          Crop  Season    Area  \
0  Tamil Nadu  Ariyalur  2015 - 2016         Bajra  Kharif   649.0   
1  Tamil Nadu  Ariyalur  2015 - 2016         Bajra    Rabi    59.0   
2  Tamil Nadu  Ariyalur  2015 - 2016   Castor seed  Kharif    34.0   
3  Tamil Nadu  Ariyalur  2015 - 2016  Cotton(lint)  Kharif  8907.0   
4  Tamil Nadu  Ariyalur  2015 - 2016  Cotton(lint)    Rabi   282.0   

   Production  Yield  
0      2305.0   3.55  
1       144.0   2.44  
2        11.0   0.32  
3     12732.0   1.43  
4       836.0   2.96  

Rows : 7563
Districts : 38
Crops : 25
Seasons : <StringArray>
['Kharif', 'Rabi', 'Autumn', 'Winter']
Length: 4, dtype: str
